# MUSAN + проверка шумоподавления в Colab

Запускай ячейки сверху вниз. Ноутбук сохраняет шумы MUSAN в Google Drive, использует русскую речь M-AILABS для 100 тестовых пар и обрабатывает один пример DeepFilterNet. Уже готовые данные при повторном запуске пропускаются.

Нужны доступ к архиву MUSAN по ссылке ниже и `ru_RU.tgz` в `MyDrive/audio-harness/raw/`. Если речь уже распакована, архив не нужен. Модель может скачать веса при первом запуске. Большие файлы остаются на твоём Drive.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys
import subprocess
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'gdown', 'soundfile', 'scipy'], check=True)


In [ ]:
from pathlib import Path

DATA_ROOT = Path('/content/drive/MyDrive/audio-harness')
RAW = DATA_ROOT / 'raw'
MUSAN_DIR = RAW / 'musan'
NOISE_DIR = MUSAN_DIR / 'musan' / 'noise'
MAILABS_DIR = RAW / 'm_ailabs_ru'
EVAL_ROOT = DATA_ROOT / 'derived' / 'denoise_eval'
MUSAN_FILE_ID = '1kngNXIxSjXTV2JtS9PAbE03yQcnx7iu_'
MAILABS_ARCHIVE = RAW / 'ru_RU.tgz'

SEED = 42
PAIR_COUNT = 100
TEST_BOOK = ('female', 'hajdurova', 'mnogo_shuma_iz_nichego')

RAW.mkdir(parents=True, exist_ok=True)
print('Drive:', DATA_ROOT)


## 1. MUSAN: скачать и достать только шумы

Архив определяется по содержимому (ZIP или TAR), а не по имени. WAV распаковываются в одну предсказуемую папку. Нераспознанный или неполный архив остаётся на Drive для диагностики; исходный архив удаляется лишь после успешной проверки результата.


In [ ]:
import shutil
import tarfile
import zipfile

ARCHIVE = MUSAN_DIR / 'musan_archive'
MUSAN_DIR.mkdir(parents=True, exist_ok=True)

def noise_relative(name):
    parts = Path(name.replace('\\', '/')).parts
    if 'noise' not in parts:
        return None
    tail = parts[parts.index('noise') + 1:]
    if not tail or any(part in ('', '.', '..') for part in tail):
        return None
    if not tail[-1].lower().endswith('.wav'):
        return None
    return Path(*tail)

def extract_noise(archive_path):
    count = 0
    if zipfile.is_zipfile(archive_path):
        with zipfile.ZipFile(archive_path) as archive:
            for info in archive.infolist():
                rel = noise_relative(info.filename)
                if rel is None or info.is_dir():
                    continue
                target = NOISE_DIR / rel
                target.parent.mkdir(parents=True, exist_ok=True)
                with archive.open(info) as src, target.open('wb') as dst:
                    shutil.copyfileobj(src, dst)
                count += 1
    elif tarfile.is_tarfile(archive_path):
        with tarfile.open(archive_path, 'r:*') as archive:
            for info in archive:
                rel = noise_relative(info.name)
                if rel is None or not info.isfile():
                    continue
                target = NOISE_DIR / rel
                target.parent.mkdir(parents=True, exist_ok=True)
                with archive.extractfile(info) as src, target.open('wb') as dst:
                    shutil.copyfileobj(src, dst)
                count += 1
    else:
        raise RuntimeError(f'Архив не распознан: {archive_path}')
    return count

existing_noise = sorted(NOISE_DIR.rglob('*.wav'))
if len(existing_noise) >= 900:
    print(f'MUSAN уже распакован: {len(existing_noise)} WAV в {NOISE_DIR}')
else:
    legacy = MUSAN_DIR / 'musan.zip'
    if not ARCHIVE.exists() and legacy.exists():
        ARCHIVE = legacy
    if ARCHIVE.exists() and not (zipfile.is_zipfile(ARCHIVE) or tarfile.is_tarfile(ARCHIVE)):
        print('Неполный архив найден; скачиваю заново:', ARCHIVE)
        ARCHIVE.unlink()
    if not ARCHIVE.exists():
        import gdown
        url = f'https://drive.google.com/file/d/{MUSAN_FILE_ID}/view'
        print('Скачиваю MUSAN из Google Drive…')
        partial = MUSAN_DIR / 'musan_archive.part'
        partial.unlink(missing_ok=True)
        gdown.download(url, str(partial), quiet=False, fuzzy=True)
        assert partial.exists() and partial.stat().st_size > 0, 'Архив MUSAN не скачался.'
        partial.replace(ARCHIVE)
    assert ARCHIVE.exists() and ARCHIVE.stat().st_size > 0, 'Архив MUSAN не скачался.'
    extracted = extract_noise(ARCHIVE)
    existing_noise = sorted(NOISE_DIR.rglob('*.wav'))
    assert extracted > 0 and existing_noise, 'В архиве не нашлись WAV из noise/.'
    print(f'Извлечено {extracted} WAV; найдено {len(existing_noise)} WAV.')
    ARCHIVE.unlink()
    print('Архив удалён после проверки распаковки.')


## 2. Русская речь M-AILABS

Если `ru_RU/by_book` уже есть, распаковка пропускается. Иначе положи `ru_RU.tgz` в `MyDrive/audio-harness/raw/` и запусти ячейку снова.


In [ ]:
BOOKS_DIR = MAILABS_DIR / 'ru_RU' / 'by_book'
if not BOOKS_DIR.is_dir():
    assert MAILABS_ARCHIVE.is_file(), f'Загрузи архив сюда: {MAILABS_ARCHIVE}'
    MAILABS_DIR.mkdir(parents=True, exist_ok=True)
    print('Распаковываю M-AILABS…')
    subprocess.run(['tar', '-xzf', str(MAILABS_ARCHIVE), '-C', str(MAILABS_DIR)], check=True)
assert BOOKS_DIR.is_dir(), f'После распаковки не найдена папка {BOOKS_DIR}'
clean_dir = BOOKS_DIR / Path(*TEST_BOOK) / 'wavs'
clean_files = sorted(clean_dir.glob('*.wav'))
noise_files = sorted(NOISE_DIR.rglob('*.wav'))
assert len(clean_files) >= PAIR_COUNT, f'Нужно {PAIR_COUNT} WAV в {clean_dir}, найдено {len(clean_files)}'
assert noise_files, f'Не найдены шумы в {NOISE_DIR}'
print('Test-книга:', '/'.join(TEST_BOOK))
print('Чистых WAV:', len(clean_files), '| шумов MUSAN:', len(noise_files))


## 3. Зафиксировать тестовые пары

100 пар создаются один раз в `derived/denoise_eval`. Выбор файлов и SNR (0, 5, 10 dB) фиксируется seed. После создания набора не меняй параметры для сравнения моделей. При повторном запуске ячейка проверит все файлы из manifest и пропустит генерацию.


In [ ]:
import csv
import random
import numpy as np
import soundfile as sf
from scipy.signal import resample_poly
from math import gcd

MANIFEST = EVAL_ROOT / 'manifest.csv'

def mono(audio):
    return audio.mean(axis=1) if audio.ndim == 2 else audio

def resample(audio, old_rate, new_rate):
    if old_rate == new_rate:
        return audio
    common = gcd(old_rate, new_rate)
    return resample_poly(audio, new_rate // common, old_rate // common).astype(np.float32)

def rms(audio):
    return float(np.sqrt(np.mean(audio.astype(np.float64) ** 2)))

def manifest_complete():
    if not MANIFEST.is_file():
        return False
    with MANIFEST.open(encoding='utf-8', newline='') as handle:
        rows = list(csv.DictReader(handle))
    return len(rows) == PAIR_COUNT and all(
        all((EVAL_ROOT / row[k]).is_file() for k in ('clean', 'noise', 'noisy'))
        for row in rows
    )

if manifest_complete():
    print('Готовый набор найден:', MANIFEST)
else:
    rng = random.Random(SEED)
    selected = rng.sample(clean_files, PAIR_COUNT)
    snrs = ([0, 5, 10] * ((PAIR_COUNT + 2) // 3))[:PAIR_COUNT]
    rng.shuffle(snrs)
    for folder in ('clean', 'noise', 'noisy'):
        (EVAL_ROOT / folder).mkdir(parents=True, exist_ok=True)

    rows = []
    for i, (clean_path, snr) in enumerate(zip(selected, snrs)):
        clean, sample_rate = sf.read(clean_path, dtype='float32')
        clean = mono(clean)
        assert len(clean) > 0 and rms(clean) > 0, f'Пустая запись: {clean_path}'
        noise_path = rng.choice(noise_files)
        noise, noise_rate = sf.read(noise_path, dtype='float32')
        noise = resample(mono(noise), noise_rate, sample_rate)
        assert len(noise) > 0 and rms(noise) > 0, f'Пустой шум: {noise_path}'
        if len(noise) >= len(clean):
            start = rng.randrange(len(noise) - len(clean) + 1)
            noise = noise[start:start + len(clean)].copy()
        else:
            noise = np.tile(noise, (len(clean) + len(noise) - 1) // len(noise))[:len(clean)]
        noise *= rms(clean) / rms(noise) / (10 ** (snr / 20))
        noisy = clean + noise
        peak = max(1.0, float(np.max(np.abs(noisy))))
        clean, noise, noisy = clean / peak, noise / peak, noisy / peak

        item_id = f'{i:04d}_snr{snr}'
        paths = {kind: f'{kind}/{item_id}.wav' for kind in ('clean', 'noise', 'noisy')}
        for kind, audio in (('clean', clean), ('noise', noise), ('noisy', noisy)):
            sf.write(EVAL_ROOT / paths[kind], audio, sample_rate, subtype='PCM_16')
        rows.append({
            'id': item_id, **paths, 'snr_db': snr,
            'source_clean': str(clean_path.relative_to(MAILABS_DIR)),
            'source_noise': str(noise_path.relative_to(NOISE_DIR)),
        })

    temp_manifest = MANIFEST.with_suffix('.csv.part')
    with temp_manifest.open('w', encoding='utf-8', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=rows[0].keys())
        writer.writeheader()
        writer.writerows(rows)
    temp_manifest.replace(MANIFEST)
    print('Создано пар:', len(rows), '| manifest:', MANIFEST)


## 4. DeepFilterNet в Python 3.11

Пакет ставится в отдельное окружение `/content/df311`. Следующая ячейка запускает модель **этим же интерпретатором**: импорт `df` в ядре Colab не нужен.


In [ ]:
DF_PYTHON = Path('/content/df311/bin/python')
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'uv'], check=True)
if not DF_PYTHON.is_file():
    subprocess.run(['uv', 'venv', '--python', '3.11', '/content/df311'], check=True)
subprocess.run([
    'uv', 'pip', 'install', '--python', str(DF_PYTHON),
    'torch==2.3.1+cpu', 'torchaudio==2.3.1+cpu',
    '--index-url', 'https://download.pytorch.org/whl/cpu',
], check=True)
subprocess.run([
    'uv', 'pip', 'install', '--python', str(DF_PYTHON),
    'DeepFilterNet==0.5.6', 'soundfile',
], check=True)
subprocess.run([str(DF_PYTHON), '-c', 'import df, torch; print("DeepFilterNet OK; torch", torch.__version__)'], check=True)


In [ ]:
import textwrap
from IPython.display import Audio, display

runner = Path('/content/run_deepfilter_one.py')
runner.write_text(textwrap.dedent('''    import sys
    from pathlib import Path
    from df.enhance import enhance, init_df, load_audio, save_audio

    source = Path(sys.argv[1])
    destination = Path(sys.argv[2])
    model, state, _ = init_df(log_file=None)
    audio, _ = load_audio(source, sr=state.sr())
    enhanced = enhance(model, state, audio)
    save_audio(destination, enhanced, state.sr())
    print('Сохранено:', destination)
'''), encoding='utf-8')

with MANIFEST.open(encoding='utf-8', newline='') as handle:
    item = next(csv.DictReader(handle))
noisy_path = EVAL_ROOT / item['noisy']
denoised_path = EVAL_ROOT / 'denoised_0000.wav'
subprocess.run([str(DF_PYTHON), str(runner), str(noisy_path), str(denoised_path)], check=True)

print('SNR:', item['snr_db'], 'dB')
for title, path in (
    ('Clean', EVAL_ROOT / item['clean']),
    ('Noisy', noisy_path),
    ('Denoised', denoised_path),
):
    print(title)
    display(Audio(filename=str(path)))
